In [1]:
# FnRGNN의 “공정성 개입 강도”를, QpiGNN이 뽑은 “불확실성”에 조건부로
# 확신이 높은 샘플은 덜 건드리고, 불확실한 샘플에서만(혹은 더 강하게) 공정성 개입을 걸면 “uncertainty-conditional intervention"

### Metric & Analysis

In [11]:
import math
import torch

import numpy as np
import networkx as nx
import seaborn as sns
import scipy.sparse as sp
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import wasserstein_distance, pearsonr, spearmanr, entropy, ks_2samp, cramervonmises_2samp, ttest_ind, levene

from torch_geometric.utils import degree
from torch_geometric.utils import to_networkx, to_scipy_sparse_matrix
#################################################################################################################


In [12]:

def normalize_adj(mx):
    rowsum = np.array(mx.sum(1)).flatten()
    r_inv = np.power(rowsum, -1.0, where=rowsum != 0)
    r_mat_inv = sp.diags(r_inv)
    return r_mat_inv.dot(mx)

def homophily_ratio(edge_index, sensitive_attr):
    edge_index = edge_index.cpu().numpy()
    sens = sensitive_attr.cpu().numpy()
    same = sum(sens[u] == sens[v] for u, v in zip(*edge_index))
    return same / edge_index.shape[1]

def assortativity_coefficient(data):
    G = to_networkx(data, to_undirected=True)
    nx.set_node_attributes(G, {i: int(data.sensitive_attr[i]) for i in range(data.num_nodes)}, "sensitive")
    return nx.attribute_assortativity_coefficient(G, "sensitive")

def local_neighborhood_fairness(edge_index, sensitive_attr):
    edge_index = edge_index.cpu().numpy()
    sens = sensitive_attr.cpu().numpy()
    diffs = []

    for node in range(len(sens)):
        neighbors = edge_index[1][edge_index[0] == node]
        if len(neighbors) > 0:
            local = sens[neighbors].mean()
            global_ = sens.mean()
            diffs.append(abs(local - global_))
    return np.mean(diffs)

def degree_balance(edge_index, sensitive_attr):
    num_nodes = sensitive_attr.size(0)
    degrees = np.bincount(edge_index[0].cpu().numpy(), minlength=num_nodes)
    sens = sensitive_attr.cpu().numpy()
    group0_degrees = degrees[sens == 0]
    group1_degrees = degrees[sens == 1]
    return abs(group0_degrees.mean() - group1_degrees.mean())

def structural_bias(features, edge_index, sens, num_hops=2, alpha=0.9):
    adj = to_scipy_sparse_matrix(edge_index, num_nodes=features.size(0)).tocsr()
    adj = adj + sp.eye(adj.shape[0])
    adj = normalize_adj(adj)
    feat = features.cpu().numpy()
    feat_smooth = feat.copy()
    
    for _ in range(num_hops):
        feat_smooth = alpha * adj.dot(feat_smooth) + (1 - alpha) * feat
    
    sens = sens.cpu().numpy()
    s0 = feat_smooth[sens == 0]
    s1 = feat_smooth[sens == 1]
    emd = np.mean([wasserstein_distance(s0[:, i], s1[:, i]) for i in range(s0.shape[1])])

    return emd

def analyze_structural_bias_with_tests(data, df, sens_attr='sens', y_col='y', y_pred_col='y_pred'):
    df['degree'] = degree(data.edge_index[0], num_nodes=data.num_nodes).cpu().numpy()
    df['error'] = np.abs(df[y_col] - df[y_pred_col])
    summary = df.groupby(sens_attr).agg({
        'degree': ['mean', 'std'],
        'error': ['mean', 'std'],
        y_pred_col: ['var']
    }).round(3)

    print(summary)

    g0 = df[df[sens_attr] == 0]
    g1 = df[df[sens_attr] == 1]
    t_stat, p_ttest = ttest_ind(g0['error'], g1['error'], equal_var=False)
    w_stat, p_levene = levene(g0['error'], g1['error'])
    ks_stat, p_ks = ks_2samp(g0[y_pred_col], g1[y_pred_col])

    print(f"  • t-test:        p = {p_ttest:.4f}")
    print(f"  • Levene test:   p = {p_levene:.4f}")
    print(f"  • KS test:       p = {p_ks:.4f}")

    return {
        'summary': summary,
        'p_ttest_error_mean': p_ttest,
        'p_levene_error_var': p_levene,
        'p_ks_pred_dist': p_ks
    }

In [13]:
def accuracy(output, labels):
    output = output.squeeze()
    preds = (output>0).type_as(labels)
    correct = preds.eq(labels).double()
    correct = correct.sum()
    return correct / len(labels)

def total_variation_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

def kl_divergence(p, q):
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    return entropy(p, q)

def js_divergence(p, q):
    return jensenshannon(p, q) ** 2

def distribution_metrics(y_true_np, y_pred_np, bins=50, range=None):
    hist_true, _ = np.histogram(y_true_np, bins=bins, range=range, density=False)
    hist_pred, _ = np.histogram(y_pred_np, bins=bins, range=range, density=False)

    hist_true = hist_true.astype(np.float64)
    hist_pred = hist_pred.astype(np.float64)

    hist_true += 1e-10
    hist_pred += 1e-10
    hist_true /= hist_true.sum()
    hist_pred /= hist_pred.sum()

    return {
        'wasserstein': wasserstein_distance(y_true_np, y_pred_np),
        'kl': kl_divergence(hist_true, hist_pred),
        'js': js_divergence(hist_true, hist_pred),
        'ks': ks_2samp(y_true_np, y_pred_np).statistic,
        'cvm': cramervonmises_2samp(y_true_np, y_pred_np).statistic,
        'tv': total_variation_distance(hist_true, hist_pred)
    }

def group_distribution_metrics(y_true_np, y_pred_np, sensitive_attr_np, bins=50, range=None):
    groups = np.unique(sensitive_attr_np.astype(int))
    if len(groups) != 2:
        raise ValueError(f"필요한 그룹(0, 1) 중 일부가 누락되었습니다. 존재하는 그룹: {groups}")

    results = {}
    metrics = ['wasserstein', 'kl', 'js', 'ks', 'cvm', 'tv']
    group_metrics = {}

    for g in groups:
        idx = sensitive_attr_np == g
        group_metrics[f"group_{g}"] = distribution_metrics(y_true_np[idx], y_pred_np[idx], bins=bins, range=range)

    for m in metrics:
        try:
            diff = abs(group_metrics["group_0"][m] - group_metrics["group_1"][m])
            results[f"{m}_diff"] = diff
            results[f"{m}_g0"] = group_metrics["group_0"][m]
            results[f"{m}_g1"] = group_metrics["group_1"][m]
        except KeyError as e:
            raise KeyError(f"그룹별 지표 계산 중 '{e}' 누락. 현재 그룹들: {group_metrics.keys()}")
    return results

def fair_metric(output, labels, sens, idx):
    val_y = labels[idx].cpu().numpy()
    idx_s0 = sens.cpu().numpy()[idx.cpu().numpy()]==0
    idx_s1 = sens.cpu().numpy()[idx.cpu().numpy()]>0

    idx_s0_y1 = np.bitwise_and(idx_s0,val_y>0)
    idx_s1_y1 = np.bitwise_and(idx_s1,val_y>0)

    pred_y = (output[idx].squeeze()>0.5).type_as(labels).cpu().numpy()

    parity = abs(sum(pred_y[idx_s0])/sum(idx_s0)-sum(pred_y[idx_s1])/sum(idx_s1))
    equality = abs(sum(pred_y[idx_s0_y1])/sum(idx_s0_y1)-sum(pred_y[idx_s1_y1])/sum(idx_s1_y1))

    return parity, equality

def compute_graph_fairness_stats(data, edge_index):
    sens = data.sensitive_attr.cpu().numpy()
    labels = data.y.cpu().numpy()
    src, dst = edge_index[0].cpu().numpy(), edge_index[1].cpu().numpy()
    num_nodes = len(sens)

    print("========== Dataset Summary ==========")
    print(f"Total nodes: {num_nodes}")
    print(f"Total edges: {edge_index.shape[1]}")

    # 1. 민감 속성 분포
    g0, g1 = np.sum(sens == 0), np.sum(sens == 1)
    print("\n--- Sensitive Attribute Distribution ---")
    print(f"Group 0: {g0} ({g0/num_nodes:.2%})")
    print(f"Group 1: {g1} ({g1/num_nodes:.2%})")

    # 2. 예측값 분포
    print("\n--- Label Distribution ---")
    print(f"Overall: mean={labels.mean():.4f}, std={labels.std():.4f}")
    print(f"Group 0: mean={labels[sens==0].mean():.4f}, std={labels[sens==0].std():.4f}")
    print(f"Group 1: mean={labels[sens==1].mean():.4f}, std={labels[sens==1].std():.4f}")

    # 3. 상관관계
    pear, _ = pearsonr(sens, labels)
    spear, _ = spearmanr(sens, labels)
    print("\n--- Correlation (Sensitive vs Label) ---")
    print(f"Pearson:  {pear:.4f}")
    print(f"Spearman: {spear:.4f}")

    # 4. Homophily
    def homophily(attr):
        return np.mean(attr[src] == attr[dst])
    sens_hom = homophily(sens)
    label_hom = homophily(labels.round())  # 연속형 label인 경우 이진화
    print("\n--- Graph Homophily ---")
    print(f"Sensitive attribute homophily: {sens_hom:.4f}")
    print(f"Label homophily: {label_hom:.4f}")

    # 5. Group별 node degree
    degree = np.bincount(src, minlength=num_nodes)
    deg_g0 = degree[sens == 0]
    deg_g1 = degree[sens == 1]
    print("\n--- Node Degree (per group) ---")
    print(f"Group 0: mean={deg_g0.mean():.2f}, std={deg_g0.std():.2f}")
    print(f"Group 1: mean={deg_g1.mean():.2f}, std={deg_g1.std():.2f}")

    # 6. 이웃 구성 동질성 (각 노드 이웃 중 같은 그룹 비율 평균)
    same_group_ratios = []
    for i in range(num_nodes):
        neighbors = dst[src == i]
        if len(neighbors) > 0:
            same_ratio = np.mean(sens[neighbors] == sens[i])
            same_group_ratios.append(same_ratio)
    print("\n--- Neighborhood Composition ---")
    print(f"Average same-group neighbor ratio: {np.mean(same_group_ratios):.4f}")

def fair_metric_regression(output, labels, sens):
    y_g0 = output[sens == 0]
    y_g1 = output[sens == 1]
    
    # 그룹별 MSE 차이
    mse_g0 = mean_squared_error(labels[sens == 0].cpu().numpy(), y_g0.cpu().numpy()) if len(y_g0) > 0 else 0.0
    mse_g1 = mean_squared_error(labels[sens == 1].cpu().numpy(), y_g1.cpu().numpy()) if len(y_g1) > 0 else 0.0
    mse_diff = abs(mse_g0 - mse_g1)

    # 그룹별 MAE 차이
    mae_g0 = mean_absolute_error(labels[sens == 0].cpu().numpy(), y_g0.cpu().numpy()) if len(y_g0) > 0 else 0.0
    mae_g1 = mean_absolute_error(labels[sens == 0].cpu().numpy(), y_g0.cpu().numpy()) if len(y_g0) > 0 else 0.0
    mae_diff = abs(mae_g0 - mae_g1)

    # 그룹별 평균 차이
    mean_g0 = y_g0.mean().item() if len(y_g0) > 0 else 0.0
    mean_g1 = y_g1.mean().item() if len(y_g1) > 0 else 0.0
    mean_diff = abs(mean_g0 - mean_g1)
    
    return mse_diff, mae_diff, mean_diff

def output_fairness(preds, sens):
    p0 = preds[sens == 0].cpu().numpy()
    p1 = preds[sens == 1].cpu().numpy()

    return {
        "mean_gap": abs(p0.mean() - p1.mean()),
        "mae_gap": abs(np.mean(abs(p0 - p0.mean())) - np.mean(abs(p1 - p1.mean()))),
        "mse_gap": abs(np.mean((p0 - p0.mean()) ** 2) - np.mean((p1 - p1.mean()) ** 2)),
        "wasserstein": wasserstein_distance(p0, p1),
        "js_divergence": jensenshannon(np.histogram(p0, bins=30, density=True)[0] + 1e-10,
                                       np.histogram(p1, bins=30, density=True)[0] + 1e-10)
    }

def metric_wd(feature, adj_norm, flag, weakening_factor, max_hop):
    feature = (feature / feature.norm(dim=0)).detach().cpu().numpy()
    adj_norm = (0.5 * adj_norm + 0.5 * sparse.eye(adj_norm.shape[0])).toarray()  # lambda_{max} = 2
    emd_distances = []
    cumulation = np.zeros_like(feature)

    if max_hop == 0:
        cumulation = feature
    else:
        for i in range(max_hop):
            cumulation += pow(weakening_factor, i) * adj_norm.dot(feature)

    for i in range(feature.shape[1]):
        class_1 = cumulation[torch.eq(flag, 0), i]
        class_2 = cumulation[torch.eq(flag, 1), i]
        emd = wasserstein_distance(class_1, class_2)
        emd_distances.append(emd)

    emd_distances = [0 if math.isnan(x) else x for x in emd_distances]

    if max_hop == 0:
        print('Attribute bias : ')
    else:
        print('Structural bias : ')

    print("Sum of all Wasserstein distance value across feature dimensions: " + str(sum(emd_distances)))
    print("Average of all Wasserstein distance value across feature dimensions: " + str(np.mean(np.array(emd_distances))))

    sns.distplot(np.array(emd_distances).squeeze(), rug=True, hist=True, label='EMD value distribution')
    plt.legend()
    # plt.show()

    num_list1 = emd_distances
    x = range(len(num_list1))

    plt.bar(x, height=num_list1, width=0.4, alpha=0.8, label="Wasserstein distance on reachability")
    plt.ylabel("Wasserstein distance")
    plt.legend()
    # plt.show()

    return emd_distances

### Data

In [2]:
import os
import copy
import torch
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp

from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit

############################################################################

In [3]:
def feature_norm(features):
    min_values = features.min(axis=0)[0]
    max_values = features.max(axis=0)[0]
    return 2*(features - min_values).div(max_values-min_values) - 1

def sp2sptensor(m):
    sparse_m = sp.coo_matrix(m).astype(np.float64)
    indices = torch.from_numpy(np.vstack((sparse_m.row, sparse_m.col))).long()
    values = torch.from_numpy(sparse_m.data)
    shape = torch.Size(sparse_m.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

def is_symmetric(m):
    res = np.int64(np.triu(m).T == np.tril(m))
    if np.where(res == 0)[0].size > 0:
        raise ValueError("The matrix is not symmetric!")
    else:
        pass

def symetric_normalize(m, half: bool):
    if not half:
        is_symmetric(m)
    else:
        m = m + m.T - np.diag(np.diagonal(m))

    hat_m = m + np.eye(m.shape[0])
    D = np.sum(hat_m, axis=1)
    D = np.diag(D)

    with np.errstate(divide='ignore'):
        D = np.power(D, -0.5)
        D[np.isinf(D)] = 0

    D[np.isinf(D)] = 0
    sn_m = np.matmul(np.matmul(D, hat_m), D)
    return sn_m


In [4]:
class German:
    def __init__(self, path):
        if not os.path.exists(path):
            raise ValueError("Data path doesn't exist!")
        else:
            self.data_path = path
        self.raw_data = self._node_process()
        self.A_tensor, self.A = self._edge_process()
        self.senIdx, self.sen_vals, self.trainIdxtensor, self.valIdxTensor, self.testIdxTensor, self.features, self.labels = self._split_data()

    def _node_process(self):
        filenames = os.listdir(self.data_path)

        for file in filenames:
            if os.path.splitext(file)[1] != '.csv':
                continue
            else:
                df_data = pd.read_csv(os.path.join(self.data_path, file))

                # modify str feature
                df_data['GoodCustomer'] = df_data['GoodCustomer'].replace(-1, 0).astype(int)
                gender_map = {'Female': 0, 'Male': 1}
                df_data['Gender'] = df_data['Gender'].map(gender_map).astype(int)

                purposeList = list(df_data['PurposeOfLoan'])
                random.shuffle(purposeList)
                purposeDict = {}
                index = 0
                for pur in purposeList:
                    if purposeDict.get(pur, None) is None:
                        purposeDict[pur] = index
                        index += 1
                    else:
                        continue

                for key in purposeDict.keys():
                    df_data['PurposeOfLoan'] = df_data['PurposeOfLoan'].map(purposeDict).fillna(-1).astype(int)
                    
                return df_data

    def _edge_process(self):
        filenames = os.listdir(self.data_path)

        for file in filenames:
            if os.path.splitext(file)[1] != '.txt':
                continue
            else:
                edges = np.loadtxt(os.path.join(self.data_path, file)).astype(int)

                # Adjacency
                num_dim = len(self.raw_data)
                A = np.zeros((num_dim, num_dim))
                for i in range(len(edges)):
                    A[edges[i][0]][edges[i][1]] = 1
                    A[edges[i][1]][edges[i][0]] = 1
                sym_norm_A = symetric_normalize(A, half=False)
                syn_norm_A_tensor = sp2sptensor(sym_norm_A)
                return syn_norm_A_tensor, A

    def _split_data(self):
        pos_data = self.raw_data[self.raw_data['GoodCustomer']==1]
        pos_index = list(pos_data.index)
        neg_data = self.raw_data[self.raw_data['GoodCustomer']==0]
        neg_index = list(neg_data.index)

        # shuffle the index
        random.seed(20)
        random.shuffle(pos_index)
        random.shuffle(neg_index)

        # split the data
        train_pos_idx = pos_index[:int(0.5*len(pos_index))]
        train_neg_idx = neg_index[:int(0.5*len(neg_index))]
        val_pos_idx = pos_index[int(0.5*len(pos_index)): int(0.75*len(pos_index))]
        val_neg_idx = neg_index[int(0.5*len(neg_index)): int(0.75*len(neg_index))]
        test_pos_idx = pos_index[int(0.75*len(pos_index)):]
        test_neg_idx = neg_index[int(0.75*len(neg_index)):]

        trainIdx = train_pos_idx + train_neg_idx
        random.shuffle(trainIdx)
        valIdx = val_pos_idx + val_neg_idx
        random.shuffle(valIdx)
        testIdx = test_pos_idx + test_neg_idx
        random.shuffle(testIdx)

        assert len(trainIdx)+len(valIdx)+len(testIdx) == len(self.raw_data), "Missing data or leaking data!"

        feature_cols = list(self.raw_data.columns)
        feature_cols.remove('GoodCustomer')
        sen_idx = feature_cols.index('Gender')
        sen_vals = self.raw_data['Gender'].values.astype(int)
        feature_data = self.raw_data[feature_cols]
        labels = self.raw_data['GoodCustomer']

        # transform to tensor
        trainIdxTensor = torch.LongTensor(trainIdx)
        valIdxTensor = torch.LongTensor(valIdx)
        testIdxTensor = torch.LongTensor(testIdx)
        featuredata = torch.FloatTensor(np.array(feature_data))
        labels = torch.LongTensor(np.array(labels))

        return sen_idx, sen_vals, trainIdxTensor, valIdxTensor, testIdxTensor, featuredata, labels

    def get_index(self):
        return [self.trainIdxtensor, self.valIdxTensor, self.testIdxTensor]

    def get_raw_data(self):
        return [self.features, self.A_tensor, self.labels]

    def generate_counterfactual_perturbation(self, data):
        feature_data = copy.deepcopy(data)
        feature_data[:, self.senIdx] = 1 - feature_data[:, self.senIdx]
        return feature_data

    def generate_node_perturbation(self, prob: float, sen: bool = False):
        feature_data = copy.deepcopy(self.features)
        r = np.random.binomial(n=1, p=prob, size=feature_data.numpy().shape)
        for i in range(len(feature_data)):
            r[i][self.senIdx] = 0
        noise = np.multiply(r, np.random.normal(0., 1., r.shape))
        noise_tensor = torch.FloatTensor(noise)
        x_hat = feature_data + noise_tensor

        if sen:
            x_hat = self.generate_counterfactual_perturbation(x_hat)
        return x_hat

    def generate_struc_perturbation(self, drop_prob: float, tensor: bool = True):
        A = copy.deepcopy(self.A)
        half_A = np.triu(A)
        row, col = np.nonzero(half_A)
        idx_perturb = np.random.binomial(n=1, p=1-drop_prob, size=row.shape)
        broken_edges = np.where(idx_perturb==0)[0]
        for idx in broken_edges:
            half_A[row[idx]][col[idx]] = 0
        new_A = symetric_normalize(half_A, half=True)
        if tensor:
            new_A = sp2sptensor(new_A)
        return new_A

def dataset_config():
    config = {
        'region_job_r': {
            'path': "./dataset/pokec",
            'dataset': 'region_job', # Pokec_z
            'predict_attr': 'completion_percentage',  # completion_percentage: 프로필을 얼마나 채웠는가 (%). 자발적 행동의 간접 지표.
            'sens_attr': 'region', # gender
            'label_number': 500,
            'sens_number': 200,
            'test_idx': False,
            'dn': 'Pokec_z_Region'
        },

        'region_job_g': {
            'path': "./dataset/pokec",
            'dataset': 'region_job',
            'predict_attr': 'completion_percentage',  # completion_percentage: 프로필을 얼마나 채웠는가 (%). 자발적 행동의 간접 지표.
            'sens_attr': 'gender', # gender
            'label_number': 500,
            'sens_number': 200,
            'test_idx': False,
            'dn': 'Pokec_z_Gender'
        },

        'region_job_2_r': {
            'path': "./dataset/pokec",
            'dataset': 'region_job_2',
            'predict_attr': 'completion_percentage',  # completion_percentage: 프로필을 얼마나 채웠는가 (%). 자발적 행동의 간접 지표.
            'sens_attr': 'region', # gender
            'label_number': 500,
            'sens_number': 200,
            'test_idx': False,
            'dn': 'Pokec_n_Region'
        },

        'region_job_2_g': {
            'path': "./dataset/pokec",
            'dataset': 'region_job_2',
            'predict_attr': 'completion_percentage',  # completion_percentage: 프로필을 얼마나 채웠는가 (%). 자발적 행동의 간접 지표.
            'sens_attr': 'gender', # gender
            'label_number': 500,
            'sens_number': 200,
            'test_idx': False,
            'dn': 'Pokec_n_Gender'
        },

        'nba_p': {
            'path': "./dataset/NBA",
            'dataset': 'nba',
            # PIE (Player Impact Estimate): 통합 퍼포먼스 지표, 퍼포먼스에 대한 차별 여부 확인 가능
            # MPG (Minutes Per Game): 출전 시간 → 팀의 코치 결정, 제도적 편향 가능성
            'predict_attr': 'PIE',
            'sens_attr': 'country',  # AGE, palyer_height, player_weight
            'label_number': 100,
            'sens_number': 50,
            'test_idx': True,
            'dn': 'NBA(PIE)_Country'
        },

        'nba_m': {
            'path': "./dataset/NBA",
            'dataset': 'nba',
            'predict_attr': 'MPG',
            'sens_attr': 'country',
            'label_number': 100,
            'sens_number': 50,
            'test_idx': True,
            'dn': 'NBA(MPG)_Country'
        },

        'german_g': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount', # LoanAmount(대출 금액), LoanRateAsPercentOfIncome(소득대비상환비율), YearsAtCurrentHome(거주연수)
            'sens_attr': 'Gender', # Gender, ForeignWorker, Single(독신여부), HasTelephone(전화기보유여부), OwnsHouse(주택소유여부), Unemployed(실직상태여부) Age(고령자로 나눠서 가능) 등등등..
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_Gender'
        },

        'german_f': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount',
            'sens_attr': 'ForeignWorker',
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_ForeignWorker'
        },

        'german_s': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount', 
            'sens_attr': 'Single', 
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_Single'
        },

        'german_t': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount', 
            'sens_attr': 'HasTelephone', 
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_HasTelephone'
        },

        'german_h': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount', 
            'sens_attr': 'OwnsHouse', 
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_OwnsHouse'
        },

        'german_e': {
            'path': './dataset/NIFTY',
            'dataset': 'german',
            'predict_attr': 'LoanAmount', 
            'sens_attr': 'Unemployed', 
            'label_number': None,
            'sens_number': None,
            'test_idx': None,
            'dn': 'German_Unemployed'
        }
    }
    return config

def load_dataset(dataset, sens_attr,predict_attr, seed, path, sens_number):
    print('Loading {} dataset from {}'.format(dataset,path))

    idx_features_labels = pd.read_csv(os.path.join(path,"{}.csv".format(dataset)))
    header = list(idx_features_labels.columns)
    header.remove("user_id")

    header.remove(sens_attr)
    header.remove(predict_attr)

    features = sp.csr_matrix(idx_features_labels[header], dtype=np.float32)
    labels = idx_features_labels[predict_attr].values
    
    idx = np.array(idx_features_labels["user_id"], dtype=int)
    idx_map = {j: i for i, j in enumerate(idx)}

    edges_unordered = np.genfromtxt(os.path.join(path, f"{dataset}_relationship.txt"), dtype=np.int64)

    mapped_edges = []
    dropped = 0
    for u, v in edges_unordered:
        u_mapped = idx_map.get(u)
        v_mapped = idx_map.get(v)
        if u_mapped is not None and v_mapped is not None:
            mapped_edges.append((u_mapped, v_mapped))
        else:
            dropped += 1

    print(f"[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: {dropped}")
    edges = np.array(mapped_edges, dtype=int)

    adj = sp.coo_matrix((np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
                        shape=(labels.shape[0], labels.shape[0]),
                        dtype=np.float32)
    
    # build symmetric adjacency matrix
    adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)

    # features = normalize(features)
    adj = adj + sp.eye(adj.shape[0])

    features = torch.FloatTensor(np.array(features.todense()))
    labels = torch.LongTensor(labels)
    
    random.seed(seed)
    label_idx = np.where(labels>=0)[0]

    # random.shuffle(label_idx)
    # idx_train = label_idx[:min(int(0.5 * len(label_idx)),label_number)]
    # idx_val = label_idx[int(0.5 * len(label_idx)):int(0.75 * len(label_idx))]
    # if test_idx:
    #     idx_test = label_idx[label_number:]
    #     idx_val = idx_test
    # else:
    #     idx_test = label_idx[int(0.75 * len(label_idx)):]

    # sens = idx_features_labels[sens_attr].values
    # sens_idx = set(np.where(sens >= 0)[0])
    # idx_test = np.asarray(list(sens_idx & set(idx_test)))
    # sens = torch.FloatTensor(sens)
    # idx_sens_train = list(sens_idx - set(idx_val) - set(idx_test))
    # random.seed(seed)
    # random.shuffle(idx_sens_train)
    # idx_sens_train = torch.LongTensor(idx_sens_train[:sens_number])

    # idx_train = torch.LongTensor(idx_train)
    # idx_val = torch.LongTensor(idx_val)
    # idx_test = torch.LongTensor(idx_test)

    # stratified sampling
    sens_all = idx_features_labels[sens_attr].values
    labels_all = labels  # np.ndarray 혹은 torch.Tensor

    valid_idx = np.where((labels_all >= 0) & (sens_all >= 0))[0]
    sens_valid = sens_all[valid_idx]
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    train_part, temp_part = next(sss1.split(valid_idx, sens_valid))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_part, test_part = next(sss2.split(temp_part, sens_valid[temp_part]))

    idx_train = torch.LongTensor(valid_idx[train_part])
    idx_val = torch.LongTensor(valid_idx[temp_part[val_part]])
    idx_test = torch.LongTensor(valid_idx[temp_part[test_part]])

    sens = torch.FloatTensor(sens_all)
    sens_idx = set(np.where(sens_all >= 0)[0])
    idx_sens_train_pool = list(sens_idx - set(idx_val.numpy()) - set(idx_test.numpy()))
    random.seed(seed)
    random.shuffle(idx_sens_train_pool)
    idx_sens_train = torch.LongTensor(idx_sens_train_pool[:sens_number])
    
    # random.shuffle(sens_idx)

    if dataset == 'nba':
        features = feature_norm(features)

    return adj, features, labels, idx_train, idx_val, idx_test, sens, idx_sens_train
  
def load_dataset_unified(dataset, sens_attr, predict_attr, seed, path="./dataset/", sens_number=500, test_idx=False):
    
    if dataset.lower() != 'german':
        full_path = path
        return load_dataset(dataset, sens_attr, predict_attr, seed, full_path, sens_number)

    elif dataset.lower() == 'german':
        full_path = path
        print('Loading {} dataset from {}'.format(dataset, full_path))
        german_data = German(full_path)
        
        adj = german_data.A_tensor
        features = german_data.features
        labels = german_data.labels
        sens = torch.FloatTensor(german_data.sen_vals)
        # idx_train, idx_val, idx_test = german_data.get_index()

        # # 민감 속성 학습용 인덱스 (val/test 제외)
        # all_idx = set(range(len(sens)))
        # sens_idx = set(np.where(sens.numpy() >= 0)[0])
        # idx_sens_train = list(sens_idx - set(idx_val.numpy()) - set(idx_test.numpy()))
        # random.seed(seed)
        # random.shuffle(idx_sens_train)
        # idx_sens_train = torch.LongTensor(idx_sens_train[:sens_number])

        # Stratified split: 50/25/25
        total_idx = np.where((labels >= 0) & (sens >= 0))[0]
        sens_np = sens[total_idx].numpy()

        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        train_idx, temp_idx = next(sss1.split(total_idx, sens_np))
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        val_idx, test_idx = next(sss2.split(temp_idx, sens_np[temp_idx]))

        idx_train = torch.LongTensor(total_idx[train_idx])
        idx_val = torch.LongTensor(total_idx[temp_idx[val_idx]])
        idx_test = torch.LongTensor(total_idx[temp_idx[test_idx]])

        sens_idx = set(np.where(sens.numpy() >= 0)[0])
        idx_sens_train = list(sens_idx - set(idx_val.numpy()) - set(idx_test.numpy()))
        random.seed(seed)
        random.shuffle(idx_sens_train)
        idx_sens_train = torch.LongTensor(idx_sens_train[:sens_number])

        return adj, features, labels, idx_train, idx_val, idx_test, sens, idx_sens_train
    else:
        raise ValueError(f"Unsupported dataset: {dataset}")

def load_and_prepare_dataset(dataset_config, config_name, seed, normalize=True):
    cfg = dataset_config.get(config_name)
    
    if cfg is None:
        raise ValueError(f"Dataset '{config_name}' not found in dataset_config.")
    
    path = cfg['path']
    dataset_name = cfg['dataset']
    predict_attr = cfg['predict_attr']
    sens_attr = cfg['sens_attr']
    label_number = cfg.get('label_number')
    sens_number = cfg.get('sens_number')
    test_idx = cfg.get('test_idx')

    df = pd.read_csv(f"{path}/{dataset_name}.csv")

    # Load graph data
    adj, features, labels, idx_train, idx_val, idx_test, sens, idx_sens_train = load_dataset_unified(
        dataset_name, sens_attr, predict_attr, seed, path, sens_number, test_idx
    )

    # Remove disconnected nodes (degree == 0)
    if isinstance(adj, torch.Tensor):
        deg = torch.sparse.sum(adj, dim=1).to_dense().numpy()
    else:
        deg = np.array(adj.sum(axis=1)).flatten()

    connected_mask = deg > 0
    connected_idx = np.where(connected_mask)[0]

    # Make new index mapping: old → new
    old_to_new = {old: new for new, old in enumerate(connected_idx)}

    # Filter node-level attributes
    features = features[connected_idx]
    labels = labels[connected_idx]
    sens = sens[connected_idx]

    # Re-map indices
    idx_train = torch.tensor([old_to_new[i.item()] for i in idx_train if i.item() in old_to_new])
    idx_val = torch.tensor([old_to_new[i.item()] for i in idx_val if i.item() in old_to_new])
    idx_test = torch.tensor([old_to_new[i.item()] for i in idx_test if i.item() in old_to_new])
    idx_sens_train = torch.tensor([old_to_new[i.item()] for i in idx_sens_train if i.item() in old_to_new])

    # Filter edge list
    if isinstance(adj, torch.Tensor):
        edge_index = adj._indices().numpy()
        src, dst = edge_index[0], edge_index[1]
    else:
        adj = adj.tocoo()
        src, dst = adj.row, adj.col

    mask = np.isin(src, connected_idx) & np.isin(dst, connected_idx)
    src_filtered = [old_to_new[i] for i in src[mask]]
    dst_filtered = [old_to_new[i] for i in dst[mask]]
    edge_index = torch.tensor([src_filtered, dst_filtered], dtype=torch.long)

    # Rebuild sparse adj
    N = len(connected_idx)
    values = np.ones(len(src_filtered))
    new_adj = sp.coo_matrix((values, (src_filtered, dst_filtered)), shape=(N, N))
    new_adj = torch.sparse_coo_tensor(
        indices=torch.tensor([src_filtered, dst_filtered]),
        values=torch.ones(len(src_filtered)),
        size=(N, N)
    )

    # Binary sensitive attribute
    if sens_attr:
        sens[sens > 0] = 1
    print(f"[{dataset_name}] sens=0: {torch.sum(sens == 0).item()}, sens=1: {torch.sum(sens == 1).item()}")
    print('-' * 50)

    # Normalize features and labels
    x_scaler, y_scaler = None, None
    if normalize:
        x_scaler = StandardScaler()
        features = torch.tensor(x_scaler.fit_transform(features.cpu()), dtype=torch.float32)

        y_scaler = StandardScaler()
        labels = torch.tensor(y_scaler.fit_transform(labels.cpu().reshape(-1, 1)), dtype=torch.float32).view(-1)

    # Build PyG Data object
    data = Data(
        x=features,
        edge_index=edge_index,
        y=labels.float(),
        sensitive_attr=sens,
        adj=new_adj  # now filtered adj
    )
    data.idx_train = idx_train
    data.idx_val = idx_val
    data.idx_test = idx_test
    data.idx_sens_train = idx_sens_train

    return data, df, cfg

def print_sensitive_attr_distribution(data_dict):
    sens = data_dict['sensitive_attr']  # torch.FloatTensor
    for split in ['idx_train', 'idx_val', 'idx_test']:
        idx = data_dict[split]
        values = sens[idx].numpy()
        unique, counts = np.unique(values, return_counts=True)
        total = len(values)
        print(f"[{split}] 총 {total}개")
        for u, c in zip(unique, counts):
            print(f"  민감속성 {int(u)}: {c}개 ({c/total:.1%})")
        print("-" * 30)


In [6]:
# Dataset List
dt_config = dataset_config()
datasets= list(dt_config.keys())

print('Dataset List:')
print(datasets)

# Dataset Load
dt_dict = {}
for ds in datasets:
    data, df, cfg = load_and_prepare_dataset(dt_config, config_name=ds, seed=1127)
    dt_dict[ds] = {
        'data': data,
        'df': df,
        'cfg': cfg
    }

# Check Dataset: Train/Val/Test 민감속성 비율 동일하게 설정
print_sensitive_attr_distribution(dt_dict['nba_p']['data'])

Dataset List:
['region_job_r', 'region_job_g', 'region_job_2_r', 'region_job_2_g', 'nba_p', 'nba_m', 'german_g', 'german_f', 'german_s', 'german_t', 'german_h', 'german_e']
Loading region_job dataset from ./dataset/pokec
[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: 0
[region_job] sens=0: 43962, sens=1: 23834
--------------------------------------------------
Loading region_job dataset from ./dataset/pokec
[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: 0
[region_job] sens=0: 34308, sens=1: 33488
--------------------------------------------------
Loading region_job_2 dataset from ./dataset/pokec
[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: 0
[region_job_2] sens=0: 47338, sens=1: 19231
--------------------------------------------------
Loading region_job_2 dataset from ./dataset/pokec
[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: 0
[region_job_2] sens=0: 34125, sens=1: 32444
--------------------------------------------------
Loading nba dataset from ./dataset/NBA
[INFO] 유효하지 않은 user_id로 인해 제거된 edge 수: 0
[nba] sens=

/tmp/ipykernel_3468100/224766008.py:11: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


[german] sens=0: 310, sens=1: 690
--------------------------------------------------
Loading german dataset from ./dataset/NIFTY
[german] sens=0: 310, sens=1: 690
--------------------------------------------------
Loading german dataset from ./dataset/NIFTY
[german] sens=0: 310, sens=1: 690
--------------------------------------------------
Loading german dataset from ./dataset/NIFTY
[german] sens=0: 310, sens=1: 690
--------------------------------------------------
Loading german dataset from ./dataset/NIFTY
[german] sens=0: 310, sens=1: 690
--------------------------------------------------
Loading german dataset from ./dataset/NIFTY
[german] sens=0: 310, sens=1: 690
--------------------------------------------------
[idx_train] 총 198개
  민감속성 0: 146개 (73.7%)
  민감속성 1: 52개 (26.3%)
------------------------------
[idx_val] 총 99개
  민감속성 0: 73개 (73.7%)
  민감속성 1: 26개 (26.3%)
------------------------------
[idx_test] 총 100개
  민감속성 0: 73개 (73.0%)
  민감속성 1: 27개 (27.0%)
----------------------

In [14]:
# Evluate Dataset: eval_dt_df
eval_dt = {}
ev_dt_list = ['region_job_r', 'region_job_g', 'region_job_2_r', 'region_job_2_g', 'nba_p', 'german_g']
for ds in ev_dt_list:
    data = dt_dict[ds]['data']
    features = data.x
    edge_index = data.edge_index
    sens = data.sensitive_attr

    eval_dt[ds] = {
        "Homophily Ratio": homophily_ratio(edge_index, sens),
        "Assortativity Coefficient": assortativity_coefficient(data),
        "Local Neighborhood Fairness": local_neighborhood_fairness(edge_index, sens),
        "Degree Balance": degree_balance(edge_index, sens),
        'Structural Bias': structural_bias(features, edge_index, sens)
    }

eval_dt_df = pd.DataFrame(eval_dt).T
print(eval_dt_df)

                Homophily Ratio  Assortativity Coefficient  \
region_job_r           0.953181                   0.901644   
region_job_g           0.479239                  -0.042879   
region_job_2_r         0.955858                   0.897273   
region_job_2_g         0.488935                  -0.022346   
nba_p                  0.728806                   0.233877   
german_g               0.809190                   0.536039   

                Local Neighborhood Fairness  Degree Balance  Structural Bias  
region_job_r                       0.404528        3.283391         0.035817  
region_job_g                       0.190116        0.922710         0.029458  
region_job_2_r                     0.370787        1.902646         0.050487  
region_job_2_g                     0.193029        0.353691         0.032298  
nba_p                              0.127263        9.834965         0.041292  
german_g                           0.251152        4.291912         0.174177  


### Model

In [19]:
import math
import torch
import argparse

import numpy as np
import seaborn as sns
import scipy.sparse as sp
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F

from torch import nn
from torch import Tensor
from scipy import sparse
from typing import Optional
from torch.nn.parameter import Parameter
from torch_geometric.typing import OptTensor
from torch_sparse import SparseTensor, matmul
from deeprobust.graph.defense.pgd import PGD, prox_operators
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv

##############################################################################

ModuleNotFoundError: No module named 'torch_scatter'

In [20]:
pip install torch_scatter

  Preparing metadata (setup.py) ... done
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp38-cp38-linux_x86_64.whl size=3813277 sha256=88d6a23969a44277decfc5ee1a8e2204652807d9f00e71e5540a17862520ab65
  Stored in directory: /home/sypark/.cache/pip/wheels/54/0c/3d/5e4aea36abfd59a43f8bb9859f545baa46c2a880a99a428db6
Successfully built torch_scatter
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def str2bool(v):
    if v.lower() in ('yes', 'true', 't', 'y', '1'):
        return True
    elif v.lower() in ('no', 'false', 'f', 'n', '0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Unsupported value encountered.')

def feature_norm(features):
    min_values = features.min(axis=0)[0]
    max_values = features.max(axis=0)[0]
    return 2*(features - min_values).div(max_values-min_values) - 1

def get_sen(sens, idx_sens_train):
    num_classes = 2  # binary sensitive attribute assumed
    one_hot = F.one_hot(sens.long(), num_classes=num_classes).float()  # (N, 2)

    # training 노드에 대해서만 정규화
    group_sums = one_hot[idx_sens_train].sum(dim=0, keepdim=True)  # (1, 2)
    group_sums[group_sums == 0] = 1  # 0으로 나누는 것 방지

    one_hot[idx_sens_train] = one_hot[idx_sens_train] / group_sums  # group-normalized

    return one_hot  # shape: (N, 2)

def quantile_loss(y_true, y_pred, tau=0.9):
    error = y_true - y_pred
    return torch.mean(torch.max(tau * error, (tau - 1) * error))

def normalize_scipy(mx):
    rowsum = np.array(mx.sum(1))
    r_inv = np.power(rowsum, -0.5).flatten()
    r_inv[np.isinf(r_inv)] = 0.
    r_mat_inv = sp.diags(r_inv)
    mx = r_mat_inv.dot(mx).dot(r_mat_inv)
    return mx

def binarize(A_debiased, adj_ori, threshold_proportion):
    the_con1 = (A_debiased - adj_ori).A
    the_con1 = np.where(the_con1 > np.max(the_con1) * threshold_proportion, 1 + the_con1 * 0, the_con1)
    the_con1 = np.where(the_con1 < np.min(the_con1) * threshold_proportion, -1 + the_con1 * 0, the_con1)
    the_con1 = np.where(np.abs(the_con1) == 1, the_con1, the_con1 * 0)
    A_debiased = adj_ori + sp.coo_matrix(the_con1)
    assert A_debiased.max() == 1
    assert A_debiased.min() == 0
    A_debiased = normalize_scipy(A_debiased)
    return A_debiased

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """Convert a scipy sparse matrix to a torch sparse tensor."""
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)


In [17]:
#FnRGNN
class GroupWiseNorm(nn.Module):
    def __init__(self):
        super(GroupWiseNorm, self).__init__()

    def forward(self, pred, sensitive_attr):
        # 민감 속성 그룹 간 분포가 다르다는 것은 bias가 있을 수 있다는 뜻이고, 이를 fairness 관점에서 줄이려는 것
        # 두 그룹의 representation의 분포를 가깝게
        mask_0 = (sensitive_attr == 0).squeeze()
        mask_1 = (sensitive_attr == 1).squeeze()
        pred_0 = pred[mask_0]
        pred_1 = pred[mask_1]
        mean_diff = torch.abs(pred_0.mean() - pred_1.mean()) if pred_0.numel() > 0 and pred_1.numel() > 0 else 0
        var_diff = torch.abs(pred_0.var() - pred_1.var()) if pred_0.numel() > 0 and pred_1.numel() > 0 else 0
        return mean_diff + var_diff

class FnRGNN(nn.Module):
    def __init__(self, nfeat, hidden_dim, dropout, lm, ld, mmd_sample, lr, weight_decay):
        super(FnRGNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lambda2 = lm
        self.lambda_dist = ld
        self.mmd_sample_size = mmd_sample

        self.gcn1 = GCNConv(nfeat, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

        self.group_norm = GroupWiseNorm()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.MSELoss()

    def compute_mmd(self, h, sensitive_attr):
        # adversarial loss 없이도 직접적인 distance loss로 추가 가능
        # 단점: 해당 속성이 뭔지 명확할 때만 쓸 수 있음. 즉, 민감 속성 라벨이 없는 경우에는 직접 쓰기 힘듦
        # 두 확률 분포 간의 평균 임베딩 차이를 측정해 그룹 간 분포가 비슷해지도록 압력
        # In-processing fairness constraints in model training: 목적: 모델이 학습 중에 민감 그룹 간 차이를 덜 반영하도록 압력 주고 싶다
        # 필요한 이유: fairness constraint를 soft하게 걸 수 있는 방법이 필요할 때, MMD는 differentiable하면서도 샘플 기반이라 유연하고 강력한 도구가 됨.
        mask_0 = (sensitive_attr == 0).squeeze()
        mask_1 = (sensitive_attr == 1).squeeze()
        h_0 = h[mask_0]
        h_1 = h[mask_1]
        
        if h_0.size(0) > self.mmd_sample_size:
            idx_0 = torch.randperm(h_0.size(0), device=h.device)[:self.mmd_sample_size]
            h_0 = h_0[idx_0]
        if h_1.size(0) > self.mmd_sample_size:
            idx_1 = torch.randperm(h_1.size(0), device=h.device)[:self.mmd_sample_size]
            h_1 = h_1[idx_1]
        
        if h_0.numel() == 0 or h_1.numel() == 0:
            return torch.tensor(0.0, device=h.device)
        
        sigma = 1.0
        xx = torch.exp(-torch.cdist(h_0, h_0) / (2 * sigma**2)).mean()
        yy = torch.exp(-torch.cdist(h_1, h_1) / (2 * sigma**2)).mean()
        xy = torch.exp(-torch.cdist(h_0, h_1) / (2 * sigma**2)).mean()
        return xx + yy - 2 * xy

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        sensitive_attr = data.sensitive_attr

        num_edges = edge_index.size(1)
        edge_weight = torch.ones(num_edges, device=x.device)
        sen_diff = sensitive_attr[edge_index[0]] != sensitive_attr[edge_index[1]]
        edge_weight[sen_diff] *= 0.5  # 민감 속성 다른 엣지 가중치 감소

        h = self.gcn1(x, edge_index, edge_weight=edge_weight)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        h = self.gcn2(h, edge_index, edge_weight=edge_weight)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        y = self.classifier(h)
        return y, h

    def optimize(self, data):
        labels, idx_train, sensitive_attr = (data.y, data.idx_train, data.sensitive_attr)
        self.train()
        self.optimizer.zero_grad()
        
        y, h= self.forward(data)
        task_loss = self.criterion(y[idx_train], labels[idx_train].unsqueeze(1).float())  # mse
        mmd_loss = self.compute_mmd(h, sensitive_attr)  # mmd
        dist_loss = self.group_norm(y, sensitive_attr)  # mean + var diff
        
        total_loss = task_loss + self.lambda2 * mmd_loss + self.lambda_dist * dist_loss
        total_loss.backward()
        self.optimizer.step()
        return total_loss.item()

In [18]:
# QpiGNN
class GQNN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, dual_output=True, fixed_margin=None):
        super().__init__()
        self.dual_output = dual_output
        self.fixed_margin = fixed_margin
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        if dual_output:
            self.fc_pred = torch.nn.Linear(hidden_dim, 1)
            if fixed_margin is None:
                self.fc_diff = torch.nn.Linear(hidden_dim, 1)
        else:
            self.fc_low = torch.nn.Linear(hidden_dim, 1)
            self.fc_upper = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        
        if self.dual_output:
            preds = self.fc_pred(x)
            
            if self.fixed_margin is not None:
                diffs = torch.ones_like(preds) * self.fixed_margin
            else:
                diffs = torch.sigmoid(self.fc_diff(x))
                
            pred_low, pred_upper = preds - diffs, preds + diffs
            
            return pred_low, pred_upper
        
        else:
            pred_low = self.fc_low(x)
            pred_upper = self.fc_upper(x)
            
            return pred_low, pred_upper
         
class GQNNLoss(nn.Module):
    def __init__(self, target_coverage=0.9, lambda_factor=0.1):
        super().__init__()
        self.target_coverage = target_coverage
        self.lf = lambda_factor

    def forward(self, preds_low, preds_upper, target):
        # 폭 계산
        diffs = (preds_upper - preds_low) / 2
        width_loss = self.lf * 2 * diffs.mean()

        # 커버리지 밖 위반 거리 계산
        below_violation = torch.relu(preds_low - target)   # y_i < lower
        above_violation = torch.relu(target - preds_upper) # y_i > upper
        total_violation = below_violation + above_violation

        # 위반된 샘플만 선택
        is_violated = (target < preds_low) | (target > preds_upper)
        violation_count = is_violated.float().sum()

        # 평균 위반 거리 (구간 밖 샘플에 대해만)
        if violation_count > 0:
            mean_violation = (total_violation * is_violated.float()).sum() / violation_count
        else:
            mean_violation = torch.tensor(0.0, device=target.device)

        covered = (preds_low <= target) & (target <= preds_upper)
        current_coverage = covered.float().mean()
        coverage_penalty = (self.target_coverage - current_coverage) ** 2

        return mean_violation + coverage_penalty + width_loss


### 실험

In [ ]:
# Experiments Setting
runs=5
epochs=500
lr=0.001
weight_decay=1e-5

cuda=torch.cuda.is_available()
device = torch.device('cuda:0' if cuda else 'cpu')
print(f"Using device: {device}")

seed=1127
np.random.seed(seed)
torch.manual_seed(seed)
if cuda:
    torch.cuda.manual_seed(seed)

model = FnRGNN(nfeat=data.x.size(1), hidden_dim=64, dropout=0.5, lm=3, ld=1, mmd_sample=500, lr=lr, weight_decay=weight_decay)

# Training
tr_dt_list = ['region_job_r', 'region_job_2_r', 'nba_p', 'nba_m', 'german_g', 'german_f', 'german_s', 'german_t', 'german_h', 'german_e']

for ds in tr_dt_list:
    data = dt_dict[ds]['data']
    data = data.to(device)

    cfg = dt_dict[ds]['cfg']
    dn = cfg['dn']


    os.makedirs(f'./FnRGNN/', exist_ok=True) 
    model_path = f'./FnRGNN/{dn}_md.pth'
    
    for run in range(runs + 1):
        model = model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = torch.nn.MSELoss()

        features = data.x.to(device).to(torch.float32)
        labels = data.y.to(device).to(torch.float32)
        sens = data.sensitive_attr.to(device).to(torch.float32)

        adj = data.adj
        if isinstance(adj, torch.Tensor):
            adj = adj.to(device).to(torch.float32)
        else:
            adj = torch.FloatTensor(adj.toarray()).to(device).to(torch.float32)

        idx_train = data.idx_train
        idx_val = data.idx_val
        idx_test = data.idx_test
        
        best_score = float('inf')
        best_model_state = None

        for epoch in range(epochs + 1):
            loss = model.optimize(data)

            # validation
            model.eval()
            with torch.no_grad():
                output, _ = model(data)

                y_true, idx_val, sensitive_attr = data.y, data.idx_val, data.sensitive_attr
                mse_val = mean_squared_error(y_true[idx_val].cpu(), output[idx_val].cpu())
                mae_val = mean_absolute_error(y_true[idx_val].cpu(), output[idx_val].cpu())
                mse_diff, mae_diff, mean_diff = fair_metric_regression(output[idx_val].cpu(), y_true[idx_val].cpu(), sensitive_attr[idx_val].cpu())
                dist_val = group_distribution_metrics(y_true[idx_val].cpu().numpy().squeeze(), output[idx_val].cpu().numpy().squeeze(), sensitive_attr[idx_val].cpu().numpy())
                
                val_score = mse_val + 0.5 * mean_diff

                if val_score < best_score:
                    best_mse = val_score
                    best_model_state = copy.deepcopy(model.state_dict())
                    torch.save(best_model_state, model_path)
                
                if epoch % 50 == 0:
                    print(f"[Run {run}, Epoch {epoch} | "
                        f"Val MSE: {mse_val:.2f}, MAE: {mae_val:.2f}, \n"
                        f"MSE Diff: {mse_diff:.2f}, MAE Diff: {mae_diff:.2f}, MEAN Diff: {mean_diff:.2f}, "
                        f"Wasserstein Diff: {dist_val['wasserstein_diff']:.2f}, JS Diff: {dist_val['js_diff']:.2f}"
                    )

# Testing
ts_dt_list = ['region_job_r', 'region_job_2_r', 'nba_p', 'nba_m', 'german_g', 'german_f', 'german_s', 'german_t', 'german_h', 'german_e']

for ds in ts_dt_list:
    data = dt_dict[ds]['data']
    data = data.to(device)

    cfg = dt_dict[ds]['cfg']
    dn = cfg['dn']

    md_results = {}
    print(f'Test FnRGNN dataset from {ds}')

    result = []
    for run in range(runs + 1):
        model = model.to(device)
        model.load_state_dict(torch.load(model_path))
        
        model.eval()
        with torch.no_grad():
            output, _ = model(data)
            y_true, idx_test, sensitive_attr = data.y, data.idx_test, data.sensitive_attr
            mse_test = mean_squared_error(y_true[idx_test].cpu(), output[idx_test].cpu())
            mae_test = mean_absolute_error(y_true[idx_test].cpu(), output[idx_test].cpu())
            mse_diff, mae_diff, mean_diff = fair_metric_regression(output[idx_test].cpu(), y_true[idx_test].cpu(), sensitive_attr[idx_test].cpu())
            dist_test = group_distribution_metrics(y_true[idx_test].cpu().numpy().squeeze(), output[idx_test].cpu().numpy().squeeze(), sensitive_attr[idx_test].cpu().numpy())
            
        result.append([mse_test, mae_test, mse_diff, mae_diff, mean_diff, dist_test])
            
    md_results[ds] = result